In [1]:
!pip install ultralytics roboflow onnx onnxsim -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.2/249.2 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 117.0 MB/s eta 0:00:00


In [2]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="nmmrbjRIkKkRTCm7qfCQ")
project = rf.workspace("bere-gcgue").project("smartphones_capstone")
dataset = project.version(5).download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to smartphones_capstone-5 in yolov8:: 100%|██████████| 6687/6687 [00:00<00:00, 7983.80it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
import os

data_yaml = f"{dataset.location}/data.yaml"

print("Dataset folder contents:")
print(os.listdir(dataset.location))

print("\nChecking data.yaml exists:")
print(os.path.exists(data_yaml))

Dataset folder contents:
['README.roboflow.txt', 'train', 'valid', 'README.dataset.txt', 'data.yaml', 'test']

Checking data.yaml exists:
True


In [4]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

results = model.train(
    data=data_yaml,
    epochs=120,
    imgsz=640,
    batch=16,
    patience=30,

    cos_lr=True,

    # Realistic smartphone augmentations
    hsv_h=0.01,
    hsv_s=0.35,
    hsv_v=0.25,

    degrees=10,
    translate=0.1,
    scale=0.3,
    fliplr=0.5,
    flipud=0.0,

    mosaic=0.7,
    close_mosaic=15,
    mixup=0.0,
    copy_paste=0.0,

    plots=True
)

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/smartphones_capstone-5/data.yaml, degrees=10, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.35, hsv_v=0.25, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=0.7, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, p

In [9]:
from pathlib import Path

best_models = sorted(
    Path("/content/runs/detect").glob("train*/weights/best.pt"),
    key=lambda p: p.stat().st_mtime
)

if not best_models:
    raise FileNotFoundError("No best.pt found. Training may not have completed.")

latest_best = best_models[-1]

print("Latest best model:", latest_best)

Latest best model: /content/runs/detect/train/weights/best.pt


In [10]:
from ultralytics import YOLO

trained_model = YOLO(str(latest_best))

metrics = trained_model.val(
    data=data_yaml,
    imgsz=640,
    plots=True
)

precision = metrics.box.mp
recall = metrics.box.mr
f1 = 2 * (precision * recall) / (precision + recall + 1e-9)

print("\n=== FINAL VALIDATION METRICS ===")
print(f"mAP50:     {metrics.box.map50 * 100:.2f}%")
print(f"mAP50-95:  {metrics.box.map * 100:.2f}%")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall:    {recall * 100:.2f}%")
print(f"F1-Score:  {f1 * 100:.2f}%")

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 101 layers, 9,416,670 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1050.1±659.6 MB/s, size: 30.6 KB)
val: Scanning /content/smartphones_capstone-5/valid/labels.cache... 193 images, 13 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 193/193 45.0Mit/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 168, len(boxes) = 230. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 3.7it/s 3.5s
                   all        193        230      0.774       0.77      0.788      0.711
  asus_rog_phone_9_pro         22         33      0.848      0.879      0.881      0.775
            iphone_16e  

In [11]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/train/weights/best.pt")
model.export(format="onnx", imgsz=640, simplify=True)

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11s summary (fused): 101 layers, 9,416,670 parameters, 0 gradients, 21.3 GFLOPs

PyTorch: starting from '/content/runs/detect/train/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 14, 8400) (18.3 MB)
requirements: Ultralytics requirements ['onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 406ms
Prepared 3 packages in 520ms
Installed 3 packages in 27ms
 + colorama==0.4.6
 + onnxruntime==1.26.0
 + onnxslim==0.1.94

requirements: AutoUpdate success ✅ 1.5s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export su

'/content/runs/detect/train/weights/best.onnx'

In [13]:
from google.colab import files

files.download("/content/yolo11_final_artifacts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>